# Create Graph Dataset

In [24]:
import sys
import os
import pickle as pkl
import pandas as pd
import torch

path = os.path.join('..', '.')
if path not in sys.path:
    sys.path.append(os.path.abspath(path))

from src.protein_graph import pncaGraph

from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

In [2]:
train_seqs = pd.read_csv('../data/real_train_sequences.csv')
test_seqs = pd.read_csv('../data/real_test_sequences.csv')

clustered_train_seqs = pd.read_csv('../data/clustered_train_sequences.csv')
clustered_test_seqs = pd.read_csv('../data/clustered_test_sequences.csv')

### Create graphs and corresponding Data objects

Using AlphaFold predicted structures.

In [ ]:
def create_graphs(
    train_structs_path, 
    test_structs_path, 
    clustered_train_seqs,
    clustered_test_seqs, 
    train_ref_seqs, 
    test_ref_seqs):

    train_output_dict = {}
    test_output_dict = {}
    
    for structs_path, seqs in zip([train_structs_path, test_structs_path], [train_ref_seqs, test_ref_seqs]):
        ds = structs_path[7:structs_path.find('_')]
        for f in tqdm(os.listdir(structs_path)):
            
            index = f[:f.find('_')]
            name = 'pnca_mut_' + ds + '_' + index

            pnca_m = pncaGraph(
                            pdb=f'{structs_path}/{f}',
                            lig_resname='PZA', 
                            self_loops=False,
                            cutoff_distance=12)
            
            metadata = seqs.iloc[[int(index)]]
            # display(metadata)
            
            mutation = metadata.mutation.values[0]

            if mutation in clustered_train_seqs.MUTATION.values:
                
                train_output_dict[name] = {
                    'graph':pnca_m, 
                    'metadata':metadata
                    }
            elif mutation in clustered_test_seqs.MUTATION.values:
                
                test_output_dict[name] = {
                    'graph':pnca_m, 
                    'metadata':metadata
                    }
            else:
                Exception('Mutation not found in clustered sequences.')
            
    return train_output_dict, test_output_dict

In [17]:
test_structs = '../pdb/test_pza'
train_structs = '../pdb/train_pza'

In [18]:
train_graph_dict, test_graph_dict = create_graphs(
    train_structs, 
    test_structs, 
    clustered_train_seqs, 
    clustered_test_seqs, 
    train_seqs, 
    test_seqs)


  0%|          | 0/464 [00:00<?, ?it/s]/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/MDAnalysis/topology/PDBParser.py:348: UserWarning: Unknown element  found for some atoms. These have been given an empty element record. If needed they can be guessed using MDAnalysis.topology.guessers.
  warnings.warn(wmsg)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/MDAnalysis/topology/guessers.py:146: UserWarning: Failed to guess the mass for the following atom types: 
  warnings.warn("Failed to guess the mass for the following atom types: {}".format(atom_type))
100%|██████████| 200/200 [00:15<00:00, 13.05it/s]


In [25]:
# attach sequence from fasta to each key in dictionary
# iterate through dict and creat one sample with gen_dataset per graph
# assign dataset object to dictionary


for sample in tqdm(test_graph_dict):
    test_graph_dict[sample]['graph'].gen_dataset(
        sequences= test_graph_dict[sample]['metadata'],
        edge_weights= 'exp',
        lambda_param=2,
        normalise=True
        )

for sample in tqdm(train_graph_dict):
    
    train_graph_dict[sample]['graph'].gen_dataset(
        sequences= train_graph_dict[sample]['metadata'],
        edge_weights= 'exp',
        lambda_param=2,
        normalise=True
        )

 50%|████▉     | 99/200 [01:09<01:11,  1.41it/s]tri_norm: face with normal vector of lenght 0
tri_norm: face with normal vector of lenght 0
 86%|████████▌ | 398/464 [04:47<00:47,  1.38it/s]tri_norm: face with normal vector of lenght 0
tri_norm: face with normal vector of lenght 0
100%|██████████| 464/464 [05:35<00:00,  1.38it/s]


In [26]:
# check no nans in features

for sample in tqdm(test_graph_dict):
    assert torch.isnan(test_graph_dict[sample]['graph'].dataset[0].x).any() == False
    
for sample in tqdm(train_graph_dict):
    assert torch.isnan(train_graph_dict[sample]['graph'].dataset[0].x).any() == False

100%|██████████| 464/464 [00:00<00:00, 76671.67it/s]


In [27]:
# Create MinMax scaler fit only on training data
from sklearn.preprocessing import MinMaxScaler

train_features = [train_graph_dict[sample]['graph'].dataset[0].x for sample in train_graph_dict]
train_feats_cat = torch.cat([x for x in train_features], dim=0)

# fit scaler and save
scaler = MinMaxScaler()
scaler.fit(train_feats_cat.numpy())


with open("../data/clustered_scaler.pkl", "wb") as f:
    pkl.dump(scaler, f)

In [28]:
# apply scaler to train and test features

for sample in train_graph_dict:
    train_graph_dict[sample]['graph'].dataset[0].x = torch.tensor(
        scaler.transform(train_graph_dict[sample]['graph'].dataset[0].x.numpy()), 
        dtype=torch.float
    )
    
for sample in test_graph_dict:
    test_graph_dict[sample]['graph'].dataset[0].x = torch.tensor(
        scaler.transform(test_graph_dict[sample]['graph'].dataset[0].x.numpy()), 
        dtype=torch.float
    )

In [29]:
graph_dict = {
    'train': train_graph_dict,
    'test': test_graph_dict
}

In [30]:
# save graph_dict as pickle  

with open('datasets/clustered_graph_dict.pkl', 'wb') as f:
    pkl.dump(graph_dict, f)